<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Código suplementar do livro <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a>, de <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Repositório de código: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

<!-- aviso-traducao-ptbr -->
<sub>
<b>Tradução não oficial para português do Brasil.</b> Este arquivo é uma obra
derivada do repositório original de Sebastian Raschka
(<a href="https://github.com/rasbt/reasoning-from-scratch">rasbt/reasoning-from-scratch</a>),
licenciado sob Apache License 2.0. Apenas o texto foi traduzido; o código
permanece inalterado. Não é uma publicação oficial da Manning e não substitui o
livro. Detalhes das convenções em <code>GLOSSARIO-TRADUCAO.md</code>.
</sub>

# Apêndice E: Batching e execução orientada a throughput

Pacotes usados neste notebook:

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",  # for download functions
    "torch",
    "tokenizers"
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.17
torch version: 2.10.0
tokenizers version: 0.21.4


- Ao longo dos capítulos principais, normalmente processamos um exemplo por vez
- Isso mantém o código compacto e mais fácil de entender
- Além disso, o código já é bem caro de rodar, então adicionar suporte a batching traria pouco benefício, dadas as limitações de hardware e de recursos
- No entanto, em certos contextos, poder rodar o código em modo de batch ainda é útil
- Este apêndice explica a ideia geral por trás da execução em batch e mostra como usá-la nos diferentes capítulos, com código do material suplementar

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-e/Appendix_E_F01_raschka.webp" width="400px">

&nbsp;
## E.1 Por que batching ajuda

- Há dois objetivos distintos de desempenho:
  - latency: com que rapidez obtemos a resposta para um único prompt;
  - throughput: quantos prompts conseguimos processar em um dado intervalo de tempo.
- A geração de exemplo único costuma ser melhor para minimizar latency e para depurar código
- O batching mira principalmente o throughput
- Se quisermos avaliar centenas de problemas no MATH-500, gerar muitas amostras de self-consistency ou treinar com muitos exemplos supervisionados, o batching pode reduzir substancialmente o tempo total de execução em hardware adequado
  - Dito isso, o batching não é necessariamente mais rápido em todo dispositivo
  - Modelos pequenos em CPUs ou algumas GPUs menos otimizadas podem não se beneficiar do batching; podemos até ter perdas de velocidade, porque o padding adicional e o overhead do batching podem anular os ganhos do paralelismo

&nbsp;
## E.2 Rodando geração em batch

- O principal obstáculo técnico no batching é que os prompts costumam ter comprimentos diferentes
- Por exemplo, um problema de matemática pode ser tokenizado em 40 tokens, enquanto outro pode virar 120 tokens
- Como os tensores no PyTorch precisam ter formatos retangulares, aplicamos padding nas sequências mais curtas para que todas caibam em um único tensor de batch

- Conceitualmente, isso torna a geração em batch muito mais difícil de implementar do que a geração com um único prompt
- No capítulo principal, usamos a classe `Qwen3Model` de `reasoning_from_scratch.qwen3` (que usa a implementação do Qwen3 explicada no apêndice C)
- Para a geração em batch, como precisamos rastrear os tokens de padding etc., há uma classe `Qwen3Model` separada em `reasoning_from_scratch.qwen3_batched` (o código-fonte pode ser visto no material suplementar em https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/qwen3_batched.py)

- Para ilustrar o uso dos utilitários de geração em batch, vamos ver um exemplo concreto
- Começamos com um exemplo de geração de texto de sequência única, parecido com o que usamos nos capítulos principais
- Aqui, aplicamos a dois prompts (`["2+2?", "3+3=6?"]`) de forma sequencial:

In [2]:
import torch

from reasoning_from_scratch.ch02 import (
    get_device,
    generate_text_basic_stream_cache,
)
from reasoning_from_scratch.ch03 import (
    load_model_and_tokenizer,
    render_prompt,
)

device = get_device()
model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False,
)

for problem in ["2+2?", "3+3=6?"]:
    prompt = render_prompt(problem)
    input_ids = torch.tensor(
        tokenizer.encode(prompt),
        dtype=torch.long,
        device=device,
    ).unsqueeze(0)

    for token in generate_text_basic_stream_cache(
        model=model,
        token_ids=input_ids,
        max_new_tokens=32,
        eos_token_id=tokenizer.eos_token_id,
    ):
        next_token_id = token.squeeze(0)
        print(tokenizer.decode(next_token_id.tolist()), end="", flush=True)

    print()

Using Apple Silicon GPU (MPS)
✓ qwen3/qwen3-0.6B-base.pth already up-to-date
 \boxed{4}
 \boxed{6}


- Abaixo, vamos usar um código parecido de `reasoning_from_scratch.qwen3_batched`, que suporta batching
- Note, porém, que a versão em batch não suporta streaming, o que significa que precisamos esperar até que todos os resultados sejam gerados antes de serem decodificados e impressos
- Aqui, a geração em batch usa left padding, que será explicado na próxima seção
- Por ora, vamos começar com um exemplo de uso para ilustrar como se usa (antes de entrarmos em como funciona internamente)

In [3]:
from reasoning_from_scratch.qwen3_batched import (
    generate_text_basic_batched_cache,
    load_model_and_tokenizer,
)

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False,
)

problems = ["2+2?", "3+3=6?"]
prompts = [render_prompt(problem) for problem in problems]
tokenized = [tokenizer.encode(p) for p in prompts]
pad_id = tokenizer.pad_token_id
max_len = max(len(t) for t in tokenized)

left_padded = [
    [pad_id] * (max_len - len(t)) + t
    for t in tokenized
]
input_ids = torch.tensor(left_padded, dtype=torch.long, device=device)

generated = generate_text_basic_batched_cache(
    model=model,
    token_ids=input_ids,
    max_new_tokens=32,
    eos_token_id=tokenizer.eos_token_id,
    pad_id=pad_id,
)

for row in generated:
    eos_pos = (row == tokenizer.eos_token_id).nonzero(as_tuple=True)[0]
    if len(eos_pos) > 0:
        row = row[:eos_pos[0]]
    print(tokenizer.decode(row.tolist()))

✓ qwen3/qwen3-0.6B-base.pth already up-to-date
 \boxed{4}
 \boxed{6}


- Como podemos ver, os resultados são exatamente os mesmos de antes
- A diferença é que estes resultados foram gerados em paralelo, via `generate_text_basic_batched_cache`
- A próxima seção explica brevemente como isso funciona por baixo dos panos

- Uma implementação de código ainda mais otimizada substitui `generate_text_basic_batched_cache` por `generate_text_basic_batched_cache_stop`
- O `generate_text_basic_batched_cache` mantém toda linha no batch ativo a cada passo de decodificação
- O `generate_text_basic_batched_cache_stop` remove as linhas finalizadas do batch de computação ativo (é mais complicado de implementar internamente, mas pode otimizar o desempenho)
- Isso está ilustrado na figura abaixo

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-e/Appendix_E_F02_raschka.webp?1" width="500px">

- Observação: no Qwen3, os tokens `<eos>` são `<|endoftext|>`, mas a figura usa `<eos>` por compactação visual

&nbsp;
## E.3 Padding e attention masks

- No modo de exemplo único, se tokenizamos um prompt curto como `"2+2?"`, podemos passá-lo ao modelo como um tensor simples de formato `(1, 4)`:
  - `input_ids = torch.tensor([[17, 10, 17, 30]])`

- Internamente, o modelo constrói uma attention mask causal padrão, de modo que cada posição só possa atender a si mesma e aos tokens anteriores
- Se você não tem familiaridade com self-attention, tenho um artigo que fornece mais contexto: https://magazine.sebastianraschka.com/p/understanding-and-coding-self-attention
- Conceitualmente, essa mask fica assim:

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-e/Appendix_E_F03_raschka.webp" width="400px">

- `1` significa "mascarado" e `0` significa "permitido"
- Assim, o primeiro token não pode olhar adiante para posições posteriores, o segundo token só pode olhar para as duas primeiras posições, e assim por diante
- Esse é o padrão de masking autorregressivo convencional

- O batching muda a situação, porque prompts diferentes costumam ter comprimentos diferentes
- Suponha que processemos `"2+2?"` junto com o prompt um pouco mais longo `"3+3=6?"`
- Como os tensores do PyTorch precisam ser retangulares, a linha mais curta precisa receber padding para igualar a mais longa
- Aqui, isso é feito com left padding:

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-e/Appendix_E_F04_raschka.webp" width="500px">

- Note que mantemos uma `attn_mask` adicional internamente; isso serve apenas para rastrear as posições com padding
- Nessa `attn_mask`, `True` significa com padding e `False` significa sem padding
- Usamos essa `attn_mask` adicional para identificar, na mask causal, os tokens que correspondem aos IDs de token de padding
- Mascarar as keys com padding e zerar as queries com padding são passos importantes para fazer o batching se comportar de forma parecida com a execução de exemplo único

- A propósito, usamos o token `<|endoftext|>`, mas isso não faz muita diferença, porque as posições de token correspondentes são ignoradas de qualquer forma

In [4]:
print(tokenizer.pad_token_id)

151643


In [5]:
print(tokenizer.decode([151643]))

<|endoftext|>


&nbsp;
## E.4 Capítulo 3: avaliação do MATH-500 em batch

- O material suplementar inclui um script para o método de avaliação implementado no capítulo 3, que podemos baixar e usar de forma parecida com o que fizemos no capítulo 6:

In [7]:
from reasoning_from_scratch.ch07 import download_from_github

download_from_github(
    "ch03/02_math500-verifier-scripts/evaluate_math500.py"
)
download_from_github(
    "ch03/01_main-chapter-code/math500_test.json",
    out="math500_test.json",
)

evaluate_math500.py: 3.5 KB
math500_test.json: 462.1 KB


- Depois, para rodá-lo, podemos executar o seguinte comando em um terminal (troque `uv run` por `python` se você não usa o uv):

```bash
uv run evaluate_math500.py \
  --dataset_size 500 \
  --which_model "reasoning"
```

- O material complementar também inclui uma versão disso para geração em batch, que aplica o método de batching discutido antes
- O download é parecido com o anterior, exceto que trocamos `evaluate_math500.py` por `evaluate_math500_batched.py`

In [8]:
download_from_github(
    "ch03/02_math500-verifier-scripts/evaluate_math500_batched.py"
)

evaluate_math500_batched.py: 8.3 KB


- O uso também é parecido com o da versão sem batch, exceto que agora fornecemos um argumento adicional `--batch_size`, para especificar quantos prompts e respostas o LLM deve processar em paralelo

```bash
uv run evaluate_math500_batched.py \
  --dataset_size 500 \
  --which_model "reasoning" \
  --batch_size 64
```

- O batch size ideal depende do que seu hardware aguenta; um batch size de 64 usa aproximadamente 23,39 GB de RAM (o script sem batch usa aproximadamente 1,84 GB de RAM)
- Vamos comparar e discutir a diferença de desempenho perto do fim do apêndice

&nbsp;
## E.5 Capítulo 4: amostragem por self-consistency em batch

- O script opcional `self_consistency_math500_batched.py`, que implementa a amostragem por self-consistency do capítulo 4, não mistura prompts diferentes em um único tensor com padding
-  Em vez disso, ele repete o mesmo prompt `num_samples` vezes e amostra várias continuações em paralelo para a votação de self-consistency
- Como toda linha parte do mesmo comprimento de prompt, esse script usa o `Qwen3Model` comum, de reasoning_from_scratch.qwen3, em vez de reasoning_from_scratch.qwen3_batched, já que não é preciso padding quando os prompts têm comprimentos iguais

- Podemos baixar o script da seguinte forma:

In [ ]:
download_from_github(
    "ch04/02_math500-inference-scaling-scripts/self_consistency_math500_batched.py"
)

- Para baixar a versão sem batch, basta remover o `"_batched"` do nome do arquivo acima
- Podemos rodar o script da seguinte forma (a sintaxe do script sem batch é idêntica)

```bash
uv run self_consistency_math500_batched.py \
  --which_model base \
  --temperature 0.9 \
  --top_p 0.9 \
  --num_samples 3 \
  --dataset_size 500 \
  --prompt_suffix "\n\nExplain step by step."
```

- Mais sobre o desempenho no fim deste apêndice

&nbsp;
## E.6 Capítulo 6: rollouts de GRPO em batch

- O self-refinement do capítulo 5 é uma técnica sequencial que, em si, não se beneficia de batching
- Seria possível rodar loops de self-refinement para várias entradas em paralelo, mas isso não é trivial de implementar e, por isso, não faz parte do material suplementar
- Em vez disso, seguimos com uma versão em batch do RLVR do capítulo 6
- No capítulo 6, usamos o mesmo prompt para os diferentes rollouts; logo, não é preciso padding aqui; assim, de forma parecida com a seção E.5, o código usa a classe `Qwen3Model` comum, de `reasoning_from_scratch.qwen3`
- Os scripts relevantes podem ser obtidos com:

In [ ]:
# Non-batched version
download_from_github(
    "ch06/02_rlvr_grpo_scripts_intro/rlvr_grpo_original_no_kl.py"
)

# Batched version
download_from_github(
    "ch06/02_rlvr_grpo_scripts_intro/rlvr_grpo_original_no_kl_batched.py"
)

# Batched version with GPU support
download_from_github(
    "ch06/02_rlvr_grpo_scripts_intro/rlvr_grpo_original_no_kl_batched_fsdp.py"
)

```bash
uv run rlvr_grpo_original_no_kl_batched.py \
  --num_rollouts 8 \
  --steps 100 \
  --batch_size 4 \
  --max_new_tokens 1024
```

- No script atual, `--batch_size` controla quantos rollouts são gerados em paralelo dentro de um step
- Isso aumenta o throughput, mas também aumenta a pressão sobre a memória, então, na prática, você pode precisar reduzir `--num_rollouts` ou `--max_new_tokens`
- Se você tem várias GPUs, a variante FSDP segue o mesmo padrão e acrescenta `--num_gpus`
- De novo, voltaremos à discussão de desempenho no fim deste apêndice
- Até o momento em que isto foi escrito, versões em batch dos scripts do capítulo 7 ainda não estão disponíveis no material suplementar, mas serão adicionadas com o tempo; conceitualmente, funcionarão de forma parecida com os scripts do capítulo 6

&nbsp;
## E.7 Capítulo 8: destilação em batch

- O capítulo 8 retorna ao estilo com padding do capítulo 3, já que os exemplos de destilação têm comprimentos diferentes de prompt e de resposta
- Você pode baixar o script e o dataset de treinamento de exemplo da seguinte forma:

In [9]:
from reasoning_from_scratch.ch08 import load_distill_data

download_from_github(
    "ch08/04_train_with_distillation/distill_batched.py"
)
_ = load_distill_data(
    partition="deepseek-r1-math-train",
    local_path="deepseek-r1-math-train.json",
)

distill_batched.py: 17.9 KB
deepseek-r1-math-train.json: 107538.0 KB


- Para a versão sem batch, remova o `"_batched"` do nome do arquivo
- Podemos rodar o script da seguinte forma:

```bash
uv run distill_batched.py \
  --data_path deepseek-r1-math-train.json \
  --dataset_size 12000 \
  --validation_size 10 \
  --epochs 2 \
  --use_think_tokens \
  --max_seq_len 1024 \
  --batch_size 4
```

&nbsp;
## E.8 Geração de sequência única versus em batch

- A tabela abaixo resume os números de tempo de execução e uso de RAM dos scripts acima

| Linha | Script                                   | Batch size | RAM      | Tempo total na H100 (min) | Tempo total no DGX Spark (min) |
|-----|------------------------------------------|------------|----------|------------------------|-----------------------------|
| 1   | evaluate_math500.py                      | -          | 1,8 GB   | 90,0                   | 174,7                       |
| 2   | evaluate_math500_batched.py              | 64         | 23,39 GB | 16,0                   | 108,4                       |
|     |                                          |            |          |                        |                             |
| 3   | self_consistency_math500.py              | -          | 1,79 GB  | 252,0                  | 340,8                       |
| 4   | self_consistency_math500_batched.py      | 3          | 2,45 GB  | 129,0                  | 243,3                       |
|     |                                          |            |          |                        |                             |
| 5   | rlvr_grpo_original_no_kl.py              | -          | 43,35 GB | 68,0                   | 63,7                        |
| 6   | rlvr_grpo_original_no_kl_batched.py      | 4          | 44,91 GB | 19,0                   | 23,1                        |
|     |                                          |            |          |                        |                             |
| 7   | distill.py                              | -          | 8,29 GB  | 10,9                   | 32,8                        |
| 8   | distill_batched.py                      | 4          | 8,34 GB  | 9,1                    | 28,2                        |